### 保有銘柄リストの作成（SBI、楽天：CSV、Y：手入力）
02_OWNlistに保存

In [5]:
# (1) インポート_標準/外部ライブラリ
import os
import csv
import re
import tempfile
from typing import Dict, List, Optional, Tuple, Iterable, Set


# (2) 設定_入出力パス/カラム/ラベル接頭辞
IDMAP_CSV = os.path.join("Data", "01_IDmap.csv")
OWNLIST_CSV = os.path.join("Data", "02_OWNlist.csv")

LABEL_PREFIX = "L_所有"
OWNLIST_HEADER = ["code", "所有者", "証券会社", "口座区分", "株数", "取得単価"]


# (3) ユーティリティ_正規化/原子的保存
# (3-1) 文字列正規化（None→""）
def _s(x) -> str:
    return "" if x is None else str(x).strip()


# (3-2) 原子的保存用の一時ファイル作成
def _atomic_write_csv(out_csv: str, header: List[str], rows: Iterable[Dict[str, str]]) -> None:
    out_dir = os.path.dirname(out_csv) or "."
    os.makedirs(out_dir, exist_ok=True)

    fd, tmp = tempfile.mkstemp(prefix="tmp_", suffix=".csv", dir=out_dir)
    os.close(fd)

    try:
        with open(tmp, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.DictWriter(f, fieldnames=header, extrasaction="ignore")
            w.writeheader()
            for r in rows:
                w.writerow({k: ("" if r.get(k) is None else str(r.get(k))) for k in header})
        os.replace(tmp, out_csv)
    finally:
        if os.path.exists(tmp):
            try:
                os.remove(tmp)
            except OSError:
                pass


# (4) 01_IDmap.csv_全列保持で読み書き（L_所有_* だけ編集）
# (4-1) IDmap読込_全列保持
def load_idmap_keep_all(path: str) -> Tuple[List[str], Dict[str, Dict[str, str]]]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"IDmapが見つかりません: {path}")

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = list(reader.fieldnames or [])
        rows_map: Dict[str, Dict[str, str]] = {}
        for row in reader:
            code = _s(row.get("code"))
            if not code:
                continue
            row2 = {k: ("" if v is None else str(v)) for k, v in row.items()}
            row2["code"] = code
            rows_map[code] = row2

    if "code" not in fieldnames:
        fieldnames = ["code"] + fieldnames

    return fieldnames, rows_map


# (4-2) IDmap保存_全列保持（列順は「既存+新規L_所有_*」）
def save_idmap_keep_all(fieldnames: List[str], rows_map: Dict[str, Dict[str, str]], out_csv: str) -> None:
    all_keys = set(fieldnames)
    for r in rows_map.values():
        all_keys.update(r.keys())

    cols = list(fieldnames)
    for c in sorted(all_keys):
        if c not in cols:
            cols.append(c)

    def _sort_key(code_str: str):
        m = re.match(r"^\d+", code_str)
        return (int(m.group(0)) if m else 10**12, code_str)

    sorted_codes = sorted(rows_map.keys(), key=_sort_key)

    def _iter_rows():
        for code in sorted_codes:
            r = rows_map[code]
            out = {c: ("" if r.get(c) is None else str(r.get(c))) for c in cols}
            out["code"] = code
            yield out

    _atomic_write_csv(out_csv, cols, _iter_rows())


# (4-3) IDmapから name 引き用マップ作成（code->name）
def load_idmap_name_map(idmap_csv: str) -> Dict[str, str]:
    _, rows_map = load_idmap_keep_all(idmap_csv)
    out: Dict[str, str] = {}
    for code, row in rows_map.items():
        out[_s(code)] = _s(row.get("name"))
    return out


# (5) 02_OWNlist.csv_保存（同一銘柄×人×口座は別行）
def save_ownlist(rows: List[Dict[str, str]], out_csv: str) -> None:
    def _sort_key(r: Dict[str, str]):
        code = _s(r.get("code"))
        m = re.match(r"^\d+", code)
        code_key = int(m.group(0)) if m else 10**12
        return (code_key, code, _s(r.get("所有者")), _s(r.get("証券会社")), _s(r.get("口座区分")))

    rows_sorted = sorted(rows, key=_sort_key)
    _atomic_write_csv(out_csv, OWNLIST_HEADER, rows_sorted)


# (6) 共通_入力レコード→ラベル名生成/レコード生成
# (6-1) ラベル名生成（例：L_所有_T_NISA / L_所有_Y_特定）
def make_owned_label(owner: str, account: str) -> str:
    return f"{LABEL_PREFIX}_{_s(owner)}_{_s(account)}"


# (6-2) 共通レコード生成（OWNlist行と同じキー）
def _make_owned_record(code, broker, account, shares, avg_cost, owner, name: str = "") -> Dict[str, str]:
    return {
        "code": _s(code),  # ★投信はファンド名なども来るので数値前提にしない
        "所有者": _s(owner),
        "証券会社": _s(broker),
        "口座区分": _s(account),
        "株数": _s(shares),
        "取得単価": _s(avg_cost),
        "name": _s(name),  # 表示用（CSV出力には使わない）
    }


# (7) 入力関数_手書き補助/SBI/楽天
# (7-1) 手書き補助（owner/broker/account + [[code,株数,取得単価]]、nameはIDmapから引く）
def manual_own(owner: str,
               broker: str,
               account: str,
               items: List[List],
               idmap_csv: str = IDMAP_CSV) -> List[Dict[str, str]]:
    name_map = load_idmap_name_map(idmap_csv)

    out: List[Dict[str, str]] = []
    for it in items:
        if not it or len(it) < 3:
            continue
        code = _s(it[0])
        shares = _s(it[1])
        avg_cost = _s(it[2])
        if code == "" or shares == "":
            continue
        name = name_map.get(code, "")
        out.append(_make_owned_record(code, broker, account, shares, avg_cost, owner, name=name))
    return out


# (7-2) SBI証券_保有銘柄CSV読込み（特定/NISA + つみたて投信対応）
def sbi_own(owner: str, csv_path: str) -> List[Dict[str, str]]:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"SBI CSVが見つかりません: {csv_path}")

    def _cell(x) -> str:
        s = "" if x is None else str(x)
        s = s.replace("\ufeff", "").replace("\u3000", " ")
        return s.strip()

    def _row_is_blank(r: List[str]) -> bool:
        return (not r) or all(_cell(c) == "" for c in r)

    def _num_only(x) -> str:
        s = _cell(x)
        m = re.findall(r"\d+", s.replace(",", ""))
        return "".join(m) if m else ""

    def _to_float(x: str) -> Optional[float]:
        try:
            if x is None:
                return None
            s = str(x).strip().replace(",", "")
            if s == "":
                return None
            return float(s)
        except Exception:
            return None

    def _account_from_section_title(title: str) -> str:
        t = _cell(title)
        if "NISA預り" in t:
            return "NISA"
        if "特定預り" in t:
            return "特定"
        return "不明"

    owned_rows: List[Dict[str, str]] = []

    current_account: str = "不明"
    stock_mode: bool = False

    # 0=未突入 / 1=見出し検出（空行待ち） / 2=ヘッダ待ち / 3=データ読み込み
    tsum_state: int = 0

    with open(csv_path, "r", encoding="cp932", errors="ignore") as f:
        reader = csv.reader(f)

        for _, row in enumerate(reader, 1):
            first = _cell(row[0]) if len(row) > 0 else ""
            row_is_blank = _row_is_blank(row)

            # セクション見出し（先頭一致）
            if first.startswith("株式（") or first.startswith("投資信託（"):
                stock_mode = False

                if first.startswith("投資信託（") and ("つみたて投資枠" in first):
                    tsum_state = 1
                else:
                    tsum_state = 0
                    current_account = _account_from_section_title(first)
                continue

            # つみたて投信（優先）
            if tsum_state > 0:
                if tsum_state == 1:
                    if row_is_blank:
                        tsum_state = 2
                    continue

                if tsum_state == 2:
                    if "ファンド名" in first:
                        tsum_state = 3
                    continue

                if tsum_state == 3:
                    if row_is_blank:
                        tsum_state = 0
                        continue

                    fund_name = _cell(row[0]) if len(row) > 0 else ""
                    kuchi_str = _cell(row[1]) if len(row) > 1 else ""
                    avg_cost_str = _cell(row[3]) if len(row) > 3 else ""

                    if "ファンド名" in fund_name:
                        continue

                    kuchi = _to_float(_num_only(kuchi_str))
                    avg_cost = _to_float(re.sub(r"[^\d.]", "", avg_cost_str.replace(",", "")))

                    if not kuchi:
                        continue

                    shares = kuchi / 10000.0  # 口数→株数
                    owned_rows.append(_make_owned_record(
                        code=fund_name,
                        broker="SBI",
                        account="NISA",  # ★NISA固定
                        shares=str(shares),
                        avg_cost="" if avg_cost is None else str(avg_cost),
                        owner=owner,
                        name=fund_name
                    ))
                    continue

            # 株式ヘッダ検出
            if "銘柄コード" in first:
                stock_mode = True
                continue

            # 株式データ行
            if stock_mode:
                code = _cell(row[0]) if len(row) > 0 else ""
                name = _cell(row[1]) if len(row) > 1 else ""
                shares = _num_only(row[2]) if len(row) > 2 else ""
                avg_cost = _num_only(row[4]) if len(row) > 4 else ""

                if not code or "銘柄コード" in code:
                    continue

                owned_rows.append(_make_owned_record(
                    code=code,
                    broker="SBI",
                    account=current_account,  # 特定 or NISA
                    shares=shares,
                    avg_cost=avg_cost,
                    owner=owner,
                    name=name
                ))
                continue

    return owned_rows


# (7-3) 楽天証券_保有銘柄CSV読込み（セクションから口座区分判定）
def rakuten_own(owner: str, csv_path: str) -> List[Dict[str, str]]:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"楽天CSVが見つかりません: {csv_path}")

    def _cell(x) -> str:
        s = "" if x is None else str(x)
        s = s.replace("\ufeff", "").replace("\u3000", " ")
        return s.strip()

    def _num_only(x) -> str:
        s = _cell(x)
        m = re.findall(r"\d+", s.replace(",", ""))
        return "".join(m) if m else ""

    def _account_from_section(line: str) -> str:
        t = _cell(line)
        if "特定" in t:
            return "特定"
        if "NISA" in t:
            return "NISA"
        return "不明"

    owned_rows: List[Dict[str, str]] = []

    current_account: str = "不明"
    mode: str = ""
    header: List[str] = []
    col_idx: Dict[str, int] = {}

    with open(csv_path, "r", encoding="cp932", errors="ignore") as f:
        reader = csv.reader(f)

        for row in reader:
            if not row:
                continue

            first = _cell(row[0]) if len(row) > 0 else ""
            if first == "":
                continue

            # セクション：■〇〇口座
            if first.startswith("■"):
                current_account = _account_from_section(first)
                mode = ""
                header = []
                col_idx = {}
                continue

            # ヘッダ：銘柄コード
            if "銘柄コード" in first:
                header = [_cell(c) for c in row]

                def _find_col(key_contains: str) -> Optional[int]:
                    for i, c in enumerate(header):
                        if key_contains in c:
                            return i
                    return None

                idx_code = _find_col("銘柄コード")
                idx_name = _find_col("銘柄名")
                idx_qty = _find_col("保有数量")
                idx_cost = _find_col("平均取得価額")

                if idx_code is None or idx_qty is None or idx_cost is None:
                    mode = ""
                    continue

                col_idx = {
                    "code": idx_code,
                    "name": idx_name if idx_name is not None else -1,
                    "qty": idx_qty,
                    "cost": idx_cost,
                }
                mode = "data"
                continue

            # データ行
            if mode == "data":
                code = _cell(row[col_idx["code"]]) if len(row) > col_idx["code"] else ""
                if not code or "銘柄コード" in code:
                    continue

                name = _cell(row[col_idx["name"]]) if col_idx["name"] >= 0 and len(row) > col_idx["name"] else ""
                qty = _num_only(row[col_idx["qty"]]) if len(row) > col_idx["qty"] else ""

                cost_raw = _cell(row[col_idx["cost"]]) if len(row) > col_idx["cost"] else ""
                cost_clean = re.sub(r"[^\d.]", "", cost_raw.replace(",", ""))  # 1,199.30 対応

                if qty == "":
                    continue

                owned_rows.append(_make_owned_record(
                    code=code,
                    broker="楽天",
                    account=current_account,
                    shares=qty,
                    avg_cost=cost_clean,
                    owner=owner,
                    name=name
                ))

    return owned_rows


# (8) 更新処理_入力レコードからIDmapとOWNlistを更新
# (8-1) IDmap更新（L_所有_* のみ編集。列が無ければ作る。その他は保持）
def update_idmap_owned_labels(idmap_csv: str, owned_rows: List[Dict[str, str]]) -> None:
    fieldnames, rows_map = load_idmap_keep_all(idmap_csv)

    needed_label_cols: Set[str] = set()
    for r in owned_rows:
        needed_label_cols.add(make_owned_label(r.get("所有者", ""), r.get("口座区分", "")))

    for lbl in sorted(needed_label_cols):
        if lbl not in fieldnames:
            fieldnames.append(lbl)

    existing_owned_label_cols = [c for c in fieldnames if c.startswith(f"{LABEL_PREFIX}_")]

    # L_所有_* を 0 にリセット
    for _, row in rows_map.items():
        for c in existing_owned_label_cols:
            row[c] = "0"

    # owned_rows に基づき 1 を立てる（IDmapに存在する code のみ）
    for r in owned_rows:
        code = _s(r.get("code"))
        if not code:
            continue
        if code not in rows_map:
            continue
        lbl = make_owned_label(r.get("所有者", ""), r.get("口座区分", ""))
        rows_map[code][lbl] = "1"

    save_idmap_keep_all(fieldnames, rows_map, idmap_csv)
    print(f"完了: IDmapの {LABEL_PREFIX}_* ラベルのみ更新しました -> {idmap_csv}")


# (8-2) OWNlist更新（この実行結果で全置換）
def update_ownlist(ownlist_csv: str, owned_rows: List[Dict[str, str]]) -> None:
    save_ownlist(owned_rows, ownlist_csv)
    print(f"完了: OWNlistを更新しました -> {ownlist_csv}")


# (9) 実行_入力統合→IDmap/OWNlist更新（手書き補助は特定/NISAの2回）
def run_update_all(owned_rows_all: List[Dict[str, str]]) -> None:
    update_idmap_owned_labels(IDMAP_CSV, owned_rows_all)
    update_ownlist(OWNLIST_CSV, owned_rows_all)


# (10) 実行例_関数呼び出し部分 + 手書き補助入力リスト（特定/NISAの2本）
if __name__ == "__main__":
    # (10-1) 手書き補助（特定）入力リスト：[[code, 株数, 取得単価], ...]
    MANUAL_Y_SBI_TOKUTEI = [
        ["1360", 5000, 184],
        ["1375", 100, 1023],
        ["2391", 100, 1268],
        ["2493", 100, 925],
        ["3926", 100, 311],
        ["6919", 100, 1755],
        ["7820", 100, 914],
    ]

    # (10-2) 手書き補助（NISA）入力リスト：[[code, 株数, 取得単価], ...]
    MANUAL_Y_SBI_NISA = [
        ["1343", 90, 1779],
        ["1488", 104, 1731],
        ["2296", 100, 5627],
        ["2556", 100, 1717],
        ["4502", 100, 4266],
        ["5401", 500, 645],
        ["5464", 300, 881],
        ["6073", 100, 1589],
        ["6301", 100, 4984],
        ["6539", 200, 1019],
        ["6919", 100, 1642],
        ["8130", 100, 2999],
        ["8584", 100, 4800],
        ["9368", 100, 884],
    ]

    # (10-3) 手書き補助（特定/NISA）
    owned_manual_tokutei = manual_own("Y", "SBI", "特定", MANUAL_Y_SBI_TOKUTEI, idmap_csv=IDMAP_CSV)
    owned_manual_nisa = manual_own("Y", "SBI", "NISA", MANUAL_Y_SBI_NISA, idmap_csv=IDMAP_CSV)

    # (10-4) SBI / 楽天 CSV 読み込み（必要に応じてパスを変更）
    SBI_CSV_T = os.path.join("Data", "Input_own", "sbi.csv")
    RAKUTEN_CSV_Y = os.path.join("Data", "Input_own", "rakuten.csv")

    owned_sbi = sbi_own("T", SBI_CSV_T)
    owned_rakuten = rakuten_own("T", RAKUTEN_CSV_Y)

    # (10-5) すべての入力を結合（この実行結果を正として上書き更新）
    owned_rows_all = []
    owned_rows_all += owned_manual_tokutei
    owned_rows_all += owned_manual_nisa
    owned_rows_all += owned_sbi
    owned_rows_all += owned_rakuten

    # (10-6) IDmap（L_所有_*のみ）と OWNlist（全置換）を更新
    run_update_all(owned_rows_all)


完了: IDmapの L_所有_* ラベルのみ更新しました -> Data\01_IDmap.csv
完了: OWNlistを更新しました -> Data\02_OWNlist.csv
